# 🛡️ Glu-Stock: 04_MONITOR_ALERT
**Phase**: Real-time Monitoring & Telegram Reporting

In [ ]:
!pip install -q pyTelegramBotAPI firebase-admin

In [ ]:
import json, os, firebase_admin, telebot
from firebase_admin import credentials, db
from datetime import datetime
from kaggle_secrets import UserSecretsClient

class KaggleInfra:
    @staticmethod
    def load_secrets():
        user_secrets = UserSecretsClient()
        return {
            "url": user_secrets.get_secret("FIREBASE_URL"),
            "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),
            "telegram": user_secrets.get_secret("TELEGRAM_TOKEN")
        }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred, {'databaseURL': secrets['url']})
        self.root_ref = db.reference("glu_stock")
        
    def get_history(self, limit=5):
        return self.root_ref.child("history").order_by_child("timestamp").limit_to_last(limit).get()
        
    def get_active_trades(self):
        trades = self.root_ref.child("trades").get()
        if not trades: return []
        return [v for v in trades.values() if v.get('status') == 'OPEN']

In [ ]:
def run_monitor():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    bot = telebot.TeleBot(secrets['telegram'])
    chat_id = "INSERT_YOUR_CHAT_ID_HERE"
    history = fb.get_history()
    active = fb.get_active_trades()
    msg = f"🛡️ **CLOUD STATUS** 🛡️\n📊 **Active**: {len(active)}\n\n📜 **Latest**:\n"
    if history:
        for k, v in history.items(): msg += f"- [{v['phase']}] {v['details'][:40]}...\n"
    try: bot.send_message(chat_id, msg, parse_mode="Markdown")
    except: print(msg)
run_monitor()